# MRP / 08 — When do large MNQ 5-minute bars happen? (descriptive, 2020–2025)

Descriptive only: no rule is selected or tested, and 2026 stays sealed. Anything that looks tradable here is a hypothesis for a new pre-registration, confirmed only on 2026.

Two definitions of "large":
- **relative**: |z| ≥ 3 (the MRP 001 volatility model). The time-of-day profile is part of σ̂, so a bar at a busy time (08:30, 09:30 New York) must move more to count. This tells you when moves are *surprising*.
- **absolute**: the bar's move in ticks is in the top 1% of all eligible bars, whatever the time. This tells you when the *biggest moves in ticks* happen.

Clock times are New York (bar open time). Marked slots: 08:30 releases (CPI, payrolls, claims, PPI, retail sales, GDP), 09:30 cash open, 10:00 releases, 14:00 / 14:30 FOMC statement and press conference (FOMC days, from the Federal Reserve calendar), 16:00 cash close.

Optional: put a release calendar at `src/equity_trading/reference/us_macro_calendar.csv` (columns `date,time_ny,event`, e.g. `2024-03-12,08:30,CPI`) and section 7 reports the large-bar rate per event type.

In [ ]:
import json, os, sys, tempfile
import datetime as dt
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("MRP_SYNTHETIC") == "1"
if SYNTHETIC:   # separate folder: the reversal smoke data has a planted reversal and no FX effect
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("MRP_REV_SYNTH_DIR") or tempfile.mkdtemp(prefix="mrp_rev_")
    os.environ["MRP_REV_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]
    os.environ.pop("DUKASCOPY_DIR", None)

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "src"))
from equity_trading import mnq_predict as P
from equity_trading import reversal as V
from equity_trading import big_bars as B

CFG = P.SYNTH_CFG if SYNTHETIC else P.MRPConfig()
D = dt.date
OUT = Path("08"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
print("engines", P.MRP_VERSION, V.REV_VERSION, B.BB_VERSION, "| synthetic" if SYNTHETIC else "| real data")
if SYNTHETIC and not (Path(os.environ["EQUITY_TRADING_DIR"]) / "Databento").exists():
    P.write_synthetic(os.environ["EQUITY_TRADING_DIR"], CFG.warmup_start, CFG.test[1], CFG, beta_jpy=0.0,
                      reversal_points=6.0, tails_df=4)


In [ ]:
K, TOP_Q = 3.0, 0.01
END = CFG.test[1]
REFITS = tuple(CFG.refits_train) + tuple(CFG.refits_test)
FIRST_FIT = (CFG.train[0], REFITS[0])
grid = P.load_grid(CFG.warmup_start, END, CFG)
sig2 = V.event_sigma(grid, FIRST_FIT, REFITS, END, CFG)
bt = V.bar_table(grid, sig2, CFG)
bt = bt[(bt["trade_date"] >= CFG.train[0]) & (bt["trade_date"] <= END)]
e = B.bar_events(bt, K, TOP_Q)
print(f"{len(e):,} eligible bars, {e['trade_date'].nunique()} CME days")
print(f"relative (|z| >= {K:g}): {int(e['big_z'].sum()):,} bars ({e['big_z'].mean():.2%})")
print(f"absolute (top {TOP_Q:.0%} in ticks, >= {e.attrs['abs_threshold_ticks']:.0f} ticks): {int(e['big_abs'].sum()):,} bars")
display(e.groupby("year")[["big_z", "big_abs"]].agg(["sum", "mean"]).round(4))
print("median move of large bars, ticks:", e.loc[e.big_z, "move_ticks"].median(), "(relative)",
      e.loc[e.big_abs, "move_ticks"].median(), "(absolute)")

## 1. Time of day (New York)

In [ ]:
rz, ra = B.slot_rates(e, "big_z"), B.slot_rates(e, "big_abs")
fig, ax = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
for a, r, t in ((ax[0], rz, f"relative: share of bars with |z| >= {K:g}"), (ax[1], ra, f"absolute: share of bars in the top {TOP_Q:.0%} by ticks")):
    x = np.arange(len(r))
    a.bar(x, r["rate"] * 100, width=1.0)
    a.axhline(r["big"].sum() / r["bars"].sum() * 100, color="k", lw=.8, ls="--")
    a.set_ylabel("% of bars"); a.set_title(t)
    for name, tm in B.MARKS.items():
        if tm in r.index:
            a.axvline(list(r.index).index(tm), color="r", lw=.6, alpha=.6)
ticks = [i for i, s in enumerate(rz.index) if s.endswith(":00")]
ax[1].set_xticks(ticks); ax[1].set_xticklabels([rz.index[i] for i in ticks], rotation=90)
plt.tight_layout(); plt.savefig(OUT / "time_of_day.png", dpi=110); plt.show()
print("top 15 slots by absolute large-bar rate:"); display(ra.sort_values("rate", ascending=False).head(15).round(4))
print("top 15 slots by relative large-bar rate:"); display(rz.sort_values("rate", ascending=False).head(15).round(4))

## 2. Scheduled-release and session slots: share of bars vs share of large bars

In [ ]:
for flag in ("big_z", "big_abs"):
    print(flag); display(B.mark_table(e, flag).round(4))

## 3. FOMC days

In [ ]:
win = e["ny_time"].isin(["14:00", "14:05", "14:10", "14:15", "14:30", "14:35", "14:40", "14:45"])
f = e[win].groupby(["fomc_day", "ny_time"])[["big_z", "big_abs", "move_ticks"]].agg({"big_z": "mean", "big_abs": "mean", "move_ticks": "median"})
display(f.unstack(0).round(3))
print("share of all absolute-large bars that fall on FOMC days:", round(e.loc[e.fomc_day, "big_abs"].sum() / e["big_abs"].sum(), 4),
      "| FOMC days are", round(e["fomc_day"].mean(), 4), "of bars")

## 4. Weekday × hour (% of bars that are large)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 3.8))
for a, flag in zip(ax, ("big_z", "big_abs")):
    h = B.weekday_hour(e, flag)
    im = a.imshow(h.to_numpy(), aspect="auto", cmap="viridis")
    a.set_yticks(range(len(h))); a.set_yticklabels(h.index); a.set_xticks(range(len(h.columns))); a.set_xticklabels(h.columns)
    a.set_xlabel("New York hour"); a.set_title(flag); plt.colorbar(im, ax=a)
plt.tight_layout(); plt.savefig(OUT / "weekday_hour.png", dpi=110); plt.show()
fri = e[(e["weekday"] == "Friday") & (e["ny_time"] == "08:30")]
print("Friday 08:30 bar: first Friday of month (payrolls proxy) vs other Fridays")
display(fri.groupby("first_friday")[["big_z", "big_abs", "move_ticks"]].agg({"big_z": "mean", "big_abs": "mean", "move_ticks": "median"}).round(3))

## 5. Clustering: do large bars come in bursts?

In [ ]:
for flag in ("big_z", "big_abs"):
    print(flag); display(B.clustering(e, flag).round(4)); display(B.per_day(e, flag).round(4))

## 6. Through time

In [ ]:
m = e.groupby(pd.to_datetime(e["trade_date"]).dt.to_period("M"))[["big_z", "big_abs"]].sum()
ax = m.plot(figsize=(14, 3.5), title="large bars per month"); ax.set_ylabel("bars")
plt.tight_layout(); plt.savefig(OUT / "per_month.png", dpi=110); plt.show()

## 7. Optional release calendar

In [ ]:
cal = B.load_calendar(REPO / "src/equity_trading/reference/us_macro_calendar.csv")
if cal is None:
    print("no calendar file: skipped (see the header cell for the format)")
else:
    for flag in ("big_z", "big_abs"):
        print(flag); display(B.calendar_hits(e, cal, flag).round(3))

In [ ]:
summary = {"engines": [P.MRP_VERSION, B.BB_VERSION], "synthetic": SYNTHETIC, "k": K, "top_q": TOP_Q,
           "abs_threshold_ticks": e.attrs["abs_threshold_ticks"],
           "marks": {f: B.mark_table(e, f).to_dict("records") for f in ("big_z", "big_abs")},
           "clustering": {f: B.clustering(e, f).to_dict("records") for f in ("big_z", "big_abs")},
           "top_slots_abs": ra.sort_values("rate", ascending=False).head(15).reset_index().to_dict("records"),
           "top_slots_rel": rz.sort_values("rate", ascending=False).head(15).reset_index().to_dict("records")}
(OUT / "summary.json").write_text(json.dumps(summary, indent=1, default=float))
rz.to_csv(OUT / "slots_relative.csv"); ra.to_csv(OUT / "slots_absolute.csv")